# 04 — Stack Ensemble

Train a meta-model on the OOF predictions from each model class.
Evaluate final stacked ensemble on the held-out test set.

In [ ]:
input_dir = "data/processed"
random_state = 42
model_names = ["linear", "gbdt", "nn"]

In [ ]:
import numpy as np
import pandas as pd
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

In [ ]:
# ── Load OOF predictions ──
oof_preds = pd.read_parquet(f"{input_dir}/oof_preds.parquet").values
y_train = pd.read_parquet(f"{input_dir}/y_train_full.parquet").values.ravel()
print(f"OOF predictions shape: {oof_preds.shape}")

In [ ]:
# ── Train meta-model ──
meta = LogisticRegression(random_state=random_state)
meta.fit(oof_preds, y_train)

print("Meta-model coefficients:")
for name, coef in zip(model_names, meta.coef_[0], strict=False):
    print(f"  {name:8s} {coef:.4f}")

In [ ]:
# ── Log to MLflow ──
mlflow.set_experiment("stacked_ensemble")
with mlflow.start_run():
    mlflow.log_metrics(
        {f"weight_{name}": coef for name, coef in zip(model_names, meta.coef_[0], strict=False)}
    )
    mlflow.sklearn.log_model(meta, "stacked_ensemble")
    print(f"Ensemble logged to MLflow run: {mlflow.active_run().info.run_id}")

In [ ]:
print("Done. Run 05_evaluate.ipynb for full evaluation on the test set.")